<a href="https://colab.research.google.com/github/Hackathon-D-Group-2026/Hackathon_D_Group_2026/blob/main/Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, RobustScaler
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from lightgbm import LGBMClassifier

RNG_SEED = 772026
GROUP_CODE = "D"
TARGET = "seasonal_vaccine"
ID_FIELD = "respondent_id"

# Distinct Crimson / Amber / Slate / Teal editorial visual theme for Group D
THEME = {
    "crimson": "#8c1d40",
    "amber": "#e87722",
    "teal": "#007a78",
    "slate": "#333f48",
    "plum": "#5c3c92",
    "sand": "#d9c3b0",
    "light_bg": "#fbf9f6",
}
sns.set_theme(style="ticks", context="notebook", font_scale=0.95)
plt.rcParams.update({
    "figure.dpi": 115,
    "figure.facecolor": "white",
    "axes.facecolor": "#fcfbfa",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#4a4a4a",
    "axes.titleweight": "bold",
    "axes.titlesize": 11,
    "grid.alpha": 0.25,
    "grid.linestyle": ":",
})

In [3]:
# Set REMOTE_BASE_URL to raw GitHub directory when running in shared Google Colab
REMOTE_BASE_URL = ""

train_data = pd.read_csv(REMOTE_BASE_URL + "dataset_A_training.csv")
test_data = pd.read_csv(REMOTE_BASE_URL + "dataset_A_testing.csv")
sub_template = pd.read_csv(REMOTE_BASE_URL + "dataset_A_example_submission.csv")

y_obs = train_data[TARGET].to_numpy(dtype=int)
n_tr, n_te = len(train_data), len(test_data)

print(f"Dataset A Training : {train_data.shape[0]:,} rows × {train_data.shape[1]} columns")
print(f"Dataset A Testing  : {test_data.shape[0]:,} rows × {test_data.shape[1]} columns")
print(f"Submission Template: {sub_template.shape[0]:,} rows | columns={list(sub_template.columns)} | target dtype={sub_template[TARGET].dtype}")

# Schema and leakage assertions
assert list(sub_template.columns) == [ID_FIELD, TARGET], "Unexpected submission columns"
assert sub_template[ID_FIELD].equals(test_data[ID_FIELD]), "Test IDs out of order vs template"
assert set(train_data.columns) - {TARGET} == set(test_data.columns), "Feature mismatch between train and test"
assert set(np.unique(y_obs)) == {0, 1}, "Target variable is not binary 0/1"
assert len(set(train_data[ID_FIELD]).intersection(set(test_data[ID_FIELD]))) == 0, "ID overlap between train and test!"

display(train_data.sample(4, random_state=RNG_SEED))

Dataset A Training : 4,756 rows × 31 columns
Dataset A Testing  : 4,749 rows × 30 columns
Submission Template: 4,749 rows | columns=['respondent_id', 'seasonal_vaccine'] | target dtype=float64


,respondent_id,flu_concern,flu_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,income_poverty,marital_status,rent_or_own,employment_status,census_msa,household_adults,household_children,employment_sector,number_chronic_med_conditions,seasonal_vaccine
1766,1767,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,...,"<= $75,000, Above Poverty",Married,Own,Not in Labor Force,Non-MSA,1.0,0.0,NaN,1,0
1746,1747,2.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Below Poverty,Not Married,Rent,Employed,"MSA, Not Principle City",0.0,0.0,mining,0,0
4336,4337,2.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,"<= $75,000, Above Poverty",Married,Own,Employed,Non-MSA,1.0,0.0,education,2,1
3415,3416,2.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,...,"> $75,000",Married,Own,Employed,"MSA, Not Principle City",1.0,0.0,wholesale,0,1
